# Análise do Mercado Polymarket

## 1. Introdução

#### Objetivo
Analisar o comportamento do mercado de previsão **"Will Bitcoin reach $1,000,000 by December 31, 2025?"**, utilizando consultas SQL sobre os dados do Polymarket disponibilizados no BigQuery.

Além da análise exploratória, serão propostos dois alertas para identificar comportamentos anormais do mercado.

#### Pergunta
Como evoluíram o volume negociado e a probabilidade implícita do mercado ao longo do tempo? É possível detectar períodos de atividade incomum por meio de alertas baseados em dados históricos? Além da análise do mercado escolhido, será avaliado se os alertas desenvolvidos podem ser aplicados a outros mercados relacionados ao Bitcoin.

#### Hipótese
Espera-se que mercados relacionados ao Bitcoin apresentem padrões recorrentes de comportamento diante de eventos relevantes. Assim, um mesmo algoritmo de detecção de anomalias, baseado em estatísticas de séries temporais, deverá ser capaz de identificar períodos atípicos em diferentes mercados sem necessidade de ajustes específicos.

## 2. Exploração

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
series_temporais = pd.read_csv("serie_temporal.csv")
estatisticas_gerais = pd.read_csv("estatisticas_gerais.csv")
dias_mais_movimentados = pd.read_csv("dias_mais_movimentados.csv")
dias_maior_variacao_preco = pd.read_csv("dias_maior_variacao_preco.csv")

### 2.1. Datasets

No BigQuery, foi feita a exploração inicial na tabela `markets` do `polymarket_optmized` para descobrir o `market_id` (516861) e outras informações sobre o mercado escolhido.
```
SELECT
    market_id,
    question,
    event_title,
    volume,
    created_at,
    end_date
FROM `even-continuity-441808-j0.polymarket_optimized.markets`
WHERE LOWER(question) LIKE '%bitcoin%'
ORDER BY volume DESC;
```
```
SELECT *
FROM `even-continuity-441808-j0.polymarket_optimized.trades`
WHERE market_id = 'SEU_MARKET_ID'
  AND DATE(trade_timestamp) BETWEEN '2025-01-01' AND '2025-12-31'
LIMIT 20;
```
A consulta com `SELECT *` fica muito pesada (~20GB) então a melhor saída é buscar soluções em outras tabelas (agregadas).

Em `polymarket_mart` temos as tabela `v_market_daily` que tem muitas informações relevantes que vão poupar consultas e processamento como `rolling_7d_avg_volume`

### 2.2. Consultas


#### Série temporal do mercado
Objetivo: Obter todas as informações diárias do mercado.
```
SELECT
    trade_date,
    daily_trade_count,
    daily_usd_volume,
    avg_trade_price,
    min_trade_price,
    max_trade_price,
    rolling_7d_avg_volume
FROM `even-continuity-441808-j0.polymarket_mart.v_market_daily`
WHERE market_id = '516861'
ORDER BY trade_date;
```



In [ ]:
series_temporais

#### Comentários

- **Quantidade de dias analisados:** 368 dias.
- **Período analisado:** De 30/12/2024 a 01/01/2026.
- **Primeiras observações:** O mercado apresentou uma probabilidade média de ~50.7%, mas com extremos tocando o limite inferior (0.001) e superior (0.999), indicando uma oscilação completa de confiança ao longo do ano.

### Estatísticas gerais
Objetivo: Obter todas as informações diárias do mercado.
```
SELECT
    MIN(trade_date) AS inicio,
    MAX(trade_date) AS fim,
    COUNT(*) AS dias,
    SUM(daily_trade_count) AS total_trades,
    SUM(daily_usd_volume) AS volume_total,
    AVG(avg_trade_price) AS preco_medio,
    MIN(min_trade_price) AS menor_preco,
    MAX(max_trade_price) AS maior_preco
FROM `even-continuity-441808-j0.polymarket_mart.v_market_daily`
WHERE market_id='516861';
```


In [ ]:
estatisticas_gerais

#### Comentários

- **Resumo do Mercado:** Ao longo de 368 dias, o mercado movimentou mais de 6.13 milhões em volume total, com 71.854 negociações realizadas.
- **Probabilidade Média:** O preço médio de negociação foi de 0.506, indicando que, na média do ano, a incerteza sobre o Bitcoin atingir $1M estava em equilíbrio próximo a 50%.
- **Amplitude:** A variação entre o preço mínimo (0.001) e o máximo (0.999) demonstra que o mercado passou por momentos de quase certeza em ambos os desfechos.

#### Dias com maior volume

Objetivo: Identificar os dias com maior movimentação financeira.

```
SELECT
    trade_date,
    daily_trade_count,
    daily_usd_volume
FROM `even-continuity-441808-j0.polymarket_mart.v_market_daily`
WHERE market_id='516861'
ORDER BY daily_usd_volume DESC
LIMIT 10;
```



In [ ]:
dias_mais_movimentados

#### Comentários

- **Pico de Volume:** O maior volume diário foi registrado em 29/12/2025 (~$640k), seguido de perto por outros dias em Dezembro.
- **Sazonalidade Final:** As 10 datas com maior volume pertencem todas ao mês de dezembro de 2025, o que confirma a hipótese de que a proximidade do fechamento do mercado (31/12) gera um surto de liquidez e apostas.
- **Atividade de Negociação:** O dia 24/12/2025 teve o maior número de negociações individuais (4.105), apesar de não ser o dia de maior volume financeiro, indicando alta participação de pequenos traders no feriado.

#### Dias com maior variação de preço

Objetivo: Identificar os dias em que a probabilidade implícita sofreu as maiores alterações.

```
WITH preco AS (
SELECT
    trade_date,
    avg_trade_price,
    LAG(avg_trade_price)
    OVER(ORDER BY trade_date) preco_anterior
FROM `even-continuity-441808-j0.polymarket_mart.v_market_daily`
WHERE market_id='516861'
)

SELECT
    trade_date,
    avg_trade_price,
    preco_anterior,
    ABS(avg_trade_price-preco_anterior) variacao
FROM preco
WHERE preco_anterior IS NOT NULL
ORDER BY variacao DESC
LIMIT 10;
```

In [ ]:
dias_maior_variacao_preco

#### Comentários

- **Maior Volatilidade:** A maior variação absoluta ocorreu em 24/05/2025, onde a probabilidade implícita saltou de ~3% para ~81% (uma variação de 0.779).
- **Eventos Significativos:** Datas em abril, maio e setembro de 2025 mostram variações superiores a 50% em um único dia, sugerindo reações a notícias macroeconômicas ou eventos específicos do ecossistema cripto.
- **Correlação:** Notavelmente, os dias de maior volatilidade de preço não coincidem com os dias de maior volume de dezembro, indicando que no meio do ano o preço era mais sensível e 'saltava' com menos volume comparado ao final do período.

## 3. Visualização dos dados

### 3.1. Volume diário

In [ ]:
# Converter data para datetime para melhor manipulação no gráfico
series_temporais['trade_date'] = pd.to_datetime(series_temporais['trade_date'])

# Configurar estilo
sns.set_theme(style="whitegrid")
plt.figure(figsize=(14, 6))

# Criar o gráfico
sns.lineplot(data=series_temporais, x="trade_date", y="daily_usd_volume", color='teal', linewidth=2)

# Ajustar títulos e labels
plt.title("Evolução do Volume Diário Negociado (USD)", fontsize=16, fontweight='bold')
plt.xlabel("Data", fontsize=12)
plt.ylabel("Volume (USD)", fontsize=12)

# Formatar o eixo X para mostrar meses
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.gca().xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45)

# Adicionar anotação no pico de volume
pico_data = series_temporais.loc[series_temporais['daily_usd_volume'].idxmax(), 'trade_date']
pico_valor = series_temporais['daily_usd_volume'].max()
plt.annotate(f'Pico: ${pico_valor:,.0f}',
             xy=(pico_data, pico_valor),
             xytext=(pico_data - pd.Timedelta(days=60), pico_valor * 0.9),
             arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=8))

plt.tight_layout()
plt.show()

#### Análise: Volume Financeiro
O volume negociado permaneceu em níveis baixos e estáveis durante a maior parte do ano, o que é típico de mercados de previsão de longo prazo. No entanto, há um aumento exponencial em **Dezembro de 2025**, culminando no pico de aproximadamente $640k no dia 29. Isso demonstra que a liquidez do mercado está diretamente ligada à proximidade do evento de resolução, com traders ajustando suas posições finais agressivamente conforme o prazo se encerra.

### 3.2. Número de negociações

In [ ]:
plt.figure(figsize=(14, 6))
sns.lineplot(data=series_temporais, x="trade_date", y="daily_trade_count", color='royalblue', linewidth=1.5)

plt.title("Número de Negociações Diárias", fontsize=16, fontweight='bold')
plt.xlabel("Data", fontsize=12)
plt.ylabel("Quantidade de Trades", fontsize=12)

plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.gca().xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

#### Análise: Frequência de Negociação
O gráfico mostra uma base constante de negociações ao longo do ano, com um salto drástico em **Dezembro de 2025**. O pico de participação (número de trades) ocorre na véspera de Natal, indicando que o interesse do varejo se intensificou conforme a data limite se aproximava, independentemente do volume financeiro total.

### 3.3. Preço médio

In [ ]:
plt.figure(figsize=(14, 6))
sns.lineplot(data=series_temporais, x="trade_date", y="avg_trade_price", color='darkorange', linewidth=2)

plt.title("Evolução da Probabilidade Implícita (Preço Médio)", fontsize=16, fontweight='bold')
plt.xlabel("Data", fontsize=12)
plt.ylabel("Preço (0 a 1)", fontsize=12)

plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.gca().xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

#### Análise: Probabilidade Implícita
A probabilidade implícita (preço) é extremamente volátil. Observamos que o mercado iniciou 2025 com otimismo (~80%), sofreu quedas severas para próximo de 0% em diversos momentos e encerrou o período em colapso total (0.001), indicando que o mercado concluiu que o Bitcoin não atingiria $1M no prazo estabelecido.

### 3.4. Preço mínimo e máximo

In [ ]:
plt.figure(figsize=(14, 6))

plt.fill_between(series_temporais['trade_date'],
                 series_temporais['min_trade_price'],
                 series_temporais['max_trade_price'],
                 color='gray', alpha=0.3, label='Intervalo Min-Max')

sns.lineplot(data=series_temporais, x="trade_date", y="avg_trade_price", color='red', label='Média')

plt.title("Dispersão de Preços: Mínimo vs Máximo", fontsize=16, fontweight='bold')
plt.xlabel("Data", fontsize=12)
plt.ylabel("Preço", fontsize=12)

plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.gca().xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45)
plt.legend()

plt.tight_layout()
plt.show()

#### Análise: Dispersão e Incerteza
O sombreamento cinza representa a amplitude entre o menor e o maior preço negociado no mesmo dia. A grande área sombreada durante a maior parte do ano revela uma **baixa convergência de opiniões**, onde traders compravam e vendiam em extremos opostos no mesmo dia. A convergência total só ocorre no final de dezembro, quando o preço estabiliza no mínimo absoluto.

## 4. Alertas

### Metodologia

Os alertas foram desenvolvidos utilizando técnicas de análise de séries temporais. Inicialmente foi escolhido o mercado "Will Bitcoin reach $1,000,000 by December 31, 2025?" para calibração dos parâmetros. Posteriormente, os mesmos algoritmos foram aplicados automaticamente a outros mercados relacionados ao Bitcoin, permitindo avaliar sua capacidade de generalização.

### Alerta 1 — Detecção de Volume Anômalo

O objetivo deste alerta é identificar dias em que o volume negociado de um mercado foi significativamente superior ao seu comportamento recente.

Inicialmente, foi realizado um teste com mercados relacionados ao Bitcoin. Foram selecionados mercados pertencentes aos eventos “What price will Bitcoin hit in 2025?” e “What price will Bitcoin hit in 2026?”.

Esses mercados apresentam a mesma temática e estrutura, permitindo verificar se o alerta pode ser aplicado a diferentes mercados sem alterações em sua lógica.

#### Teste 1 - Descobrir quais mercados relacionados ao bitcoin existem

```
WITH mercados_candidatos AS (
    SELECT
        market_id,
        ANY_VALUE(event_title) AS event_title,
        COUNT(*) AS quantidade_dias,
        MIN(trade_date) AS primeira_data,
        MAX(trade_date) AS ultima_data,
        SUM(daily_usd_volume) AS volume_total
    FROM `even-continuity-441808-j0.polymarket_mart.v_market_daily`
    WHERE
        LOWER(event_title) LIKE '%bitcoin%'
        AND (
            LOWER(event_title) LIKE '%reach%'
            OR LOWER(event_title) LIKE '%above%'
            OR LOWER(event_title) LIKE '%price%'
        )
        AND daily_usd_volume IS NOT NULL
    GROUP BY market_id
    HAVING COUNT(*) >= 14
)

SELECT
    market_id,
    event_title,
    quantidade_dias,
    primeira_data,
    ultima_data,
    ROUND(volume_total, 2) AS volume_total
FROM mercados_candidatos
ORDER BY
    quantidade_dias DESC,
    volume_total DESC
LIMIT 20;
```

#### Seleção dos mercados

A consulta de seleção encontrou 20 mercados com histórico diário suficiente.

Foram utilizados mercados dos eventos “What price will Bitcoin hit in 2025?” e “What price will Bitcoin hit in 2026?”. O conjunto inclui diferentes `market_id`, permitindo que o cálculo seja feito separadamente para cada mercado.

#### ALERTA 1 — DETECÇÃO DE VOLUME ANÔMALO

   O alerta compara o volume diário com os sete dias anteriores
   de cada mercado.

   Critério:
   - Z-Score >= 2: alerta moderado
   - Z-Score >= 3: alerta alto
   - Z-Score >= 4: alerta crítico

```
WITH base AS (

    SELECT
        market_id,
        event_title,
        trade_date,
        daily_trade_count,
        daily_usd_volume

    FROM `even-continuity-441808-j0.polymarket_mart.v_market_daily`

    WHERE
        event_title IN (
            'What price will Bitcoin hit in 2025?',
            'What price will Bitcoin hit in 2026?'
        )

        AND daily_usd_volume IS NOT NULL

        AND daily_usd_volume >= 0
),

serie_ordenada AS (

    SELECT
        market_id,
        event_title,
        trade_date,
        daily_trade_count,
        daily_usd_volume,

        ROW_NUMBER() OVER (
            PARTITION BY market_id
            ORDER BY trade_date
        ) AS numero_observacao

    FROM base
),

estatisticas_moveis AS (

    SELECT
        market_id,
        event_title,
        trade_date,
        daily_trade_count,
        daily_usd_volume,
        numero_observacao,

        COUNT(daily_usd_volume) OVER (
            PARTITION BY market_id
            ORDER BY trade_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS quantidade_dias_historico,

        AVG(daily_usd_volume) OVER (
            PARTITION BY market_id
            ORDER BY trade_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS media_volume_7d,

        STDDEV_SAMP(daily_usd_volume) OVER (
            PARTITION BY market_id
            ORDER BY trade_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS desvio_volume_7d

    FROM serie_ordenada
),

metricas AS (

    SELECT
        market_id,
        event_title,
        trade_date,
        daily_trade_count,
        daily_usd_volume,
        numero_observacao,
        quantidade_dias_historico,
        media_volume_7d,
        desvio_volume_7d,

        daily_usd_volume - media_volume_7d
            AS diferenca_volume,

        SAFE_DIVIDE(
            daily_usd_volume,
            media_volume_7d
        ) AS razao_volume,

        SAFE_DIVIDE(
            daily_usd_volume - media_volume_7d,
            desvio_volume_7d
        ) AS z_score

    FROM estatisticas_moveis
),

classificacao AS (

    SELECT
        *,

        CASE
            WHEN quantidade_dias_historico < 7
                THEN 'Histórico insuficiente'

            WHEN desvio_volume_7d IS NULL
                OR desvio_volume_7d = 0
                THEN 'Sem variação histórica'

            WHEN z_score >= 4
                THEN 'Crítico'

            WHEN z_score >= 3
                THEN 'Alto'

            WHEN z_score >= 2
                THEN 'Moderado'

            ELSE 'Normal'
        END AS nivel_alerta

    FROM metricas
),

resultado_final AS (

    SELECT
        market_id,
        event_title,
        trade_date,
        daily_trade_count,

        ROUND(
            daily_usd_volume,
            2
        ) AS daily_usd_volume,

        ROUND(
            media_volume_7d,
            2
        ) AS media_volume_7d,

        ROUND(
            desvio_volume_7d,
            2
        ) AS desvio_volume_7d,

        ROUND(
            diferenca_volume,
            2
        ) AS diferenca_volume,

        ROUND(
            razao_volume,
            2
        ) AS razao_volume,

        ROUND(
            z_score,
            2
        ) AS z_score,

        nivel_alerta

    FROM classificacao
)

SELECT
    *

FROM resultado_final

WHERE nivel_alerta IN (
    'Moderado',
    'Alto',
    'Crítico'
)

ORDER BY
    z_score DESC,
    market_id,
    trade_date;
```
Que gerou `alerta_volume.csv`.

In [ ]:
alerta_volume = pd.read_csv("alerta_volume.csv")

alerta_volume["trade_date"] = pd.to_datetime(
    alerta_volume["trade_date"]
)

alerta_volume.head()

In [ ]:
print("Total de alertas:", len(alerta_volume))

print(
    "Quantidade de mercados com alertas:",
    alerta_volume["market_id"].nunique()
)

print("\nQuantidade por nível:")

display(
    alerta_volume["nivel_alerta"]
    .value_counts()
    .rename_axis("nivel_alerta")
    .reset_index(name="quantidade")
)

In [ ]:
alertas_por_mercado = (
    alerta_volume
    .groupby(
        ["market_id", "event_title"]
    )
    .size()
    .reset_index(name="quantidade_alertas")
    .sort_values(
        "quantidade_alertas",
        ascending=False
    )
)

display(alertas_por_mercado)

In [ ]:
maiores_alertas = (
    alerta_volume[
        [
            "market_id",
            "event_title",
            "trade_date",
            "daily_usd_volume",
            "media_volume_7d",
            "razao_volume",
            "z_score",
            "nivel_alerta"
        ]
    ]
    .sort_values(
        "z_score",
        ascending=False
    )
    .head(10)
)

display(maiores_alertas)

In [ ]:
alerta_volume["trade_date"] = pd.to_datetime(
    alerta_volume["trade_date"]
)

alerta_volume["mes"] = (
    alerta_volume["trade_date"]
    .dt.to_period("M")
)

alertas_por_mes = (
    alerta_volume
    .groupby("mes")
    .size()
    .reset_index(name="quantidade_alertas")
)

alertas_por_mes["mes"] = alertas_por_mes["mes"].astype(str)

display(alertas_por_mes)

#### Análise dos resultados

O alerta identificou **884 registros de volume anômalo**, distribuídos entre **55 mercados**.

O mercado de código **516873** apresentou a maior quantidade de alertas, totalizando **49 ocorrências**.

O maior Score Z encontrado foi de **122.41**, registrado em **2025-11-02**. Nessa data, o volume negociado foi aproximadamente **105.83 vezes** superior à média dos sete dias anteriores.

A presença de alertas em diferentes `market_id` mostra que o algoritmo não depende de um único mercado. O cálculo é realizado separadamente para cada série temporal, respeitando o histórico e a escala de volume de cada mercado.

Observou-se que os mercados relacionados ao evento "What price will Bitcoin hit in 2025?" concentraram 502 alertas, enquanto os mercados equivalentes de 2026 apresentaram 382 ocorrências. Além disso, poucos mercados concentraram grande parte dos alertas, indicando que a atividade anômala não ocorre de forma homogênea entre todos os contratos.

A predominância de alertas classificados como críticos sugere que, quando ocorrem aumentos de volume, eles tendem a representar mudanças expressivas em relação ao histórico recente.

Também foi observada uma concentração temporal dos alertas entre o final de 2025 e o início de 2026, período em que os mercados se aproximam de sua data de resolução, o que é compatível com o aumento esperado da atividade de negociação.

### Alerta 2 — Detecção de Variações Anômalas de Preço

O segundo alerta identifica mudanças incomuns no preço médio diário dos mercados.

Como a tabela não possui uma coluna de preço de fechamento, foi utilizada a variável `avg_trade_price`, que representa o preço médio das negociações realizadas em cada dia.

Para tornar os mercados comparáveis, o alerta utiliza a variação percentual diária do preço médio. Em seguida, o módulo dessa variação é comparado à média e ao desvio padrão das variações observadas nos sete dias anteriores.

O cálculo é realizado separadamente para cada `market_id`.

#### ALERTA 2 — VARIAÇÃO ANÔMALA DO PREÇO MÉDIO

   Objetivo:
   Detectar dias em que a variação percentual do preço médio
   foi muito superior ao comportamento recente do mercado.

   Classificação:
   - Score Z >= 2: Moderado
   - Score Z >= 3: Alto
   - Score Z >= 4: Crítico

```
WITH base AS (

    SELECT
        market_id,
        event_title,
        trade_date,
        avg_trade_price,
        min_trade_price,
        max_trade_price,
        daily_trade_count,
        daily_usd_volume

    FROM `even-continuity-441808-j0.polymarket_mart.v_market_daily`

    WHERE
        event_title IN (
            'What price will Bitcoin hit in 2025?',
            'What price will Bitcoin hit in 2026?'
        )

        AND avg_trade_price IS NOT NULL
        AND avg_trade_price > 0
),

preco_anterior AS (

    SELECT
        market_id,
        event_title,
        trade_date,
        avg_trade_price,
        min_trade_price,
        max_trade_price,
        daily_trade_count,
        daily_usd_volume,

        LAG(avg_trade_price) OVER (
            PARTITION BY market_id
            ORDER BY trade_date
        ) AS preco_medio_anterior

    FROM base
),

variacao_diaria AS (

    SELECT
        market_id,
        event_title,
        trade_date,
        avg_trade_price,
        preco_medio_anterior,
        min_trade_price,
        max_trade_price,
        daily_trade_count,
        daily_usd_volume,

        avg_trade_price - preco_medio_anterior
            AS variacao_absoluta_preco,

        SAFE_DIVIDE(
            avg_trade_price - preco_medio_anterior,
            preco_medio_anterior
        ) AS variacao_percentual,

        ABS(
            SAFE_DIVIDE(
                avg_trade_price - preco_medio_anterior,
                preco_medio_anterior
            )
        ) AS modulo_variacao_percentual

    FROM preco_anterior

    WHERE preco_medio_anterior IS NOT NULL
        AND preco_medio_anterior > 0
),

estatisticas_moveis AS (

    SELECT
        market_id,
        event_title,
        trade_date,
        avg_trade_price,
        preco_medio_anterior,
        min_trade_price,
        max_trade_price,
        daily_trade_count,
        daily_usd_volume,
        variacao_absoluta_preco,
        variacao_percentual,
        modulo_variacao_percentual,

        COUNT(modulo_variacao_percentual) OVER (
            PARTITION BY market_id
            ORDER BY trade_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS quantidade_dias_historico,

        AVG(modulo_variacao_percentual) OVER (
            PARTITION BY market_id
            ORDER BY trade_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS media_variacao_7d,

        STDDEV_SAMP(modulo_variacao_percentual) OVER (
            PARTITION BY market_id
            ORDER BY trade_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS desvio_variacao_7d

    FROM variacao_diaria
),

metricas AS (

    SELECT
        market_id,
        event_title,
        trade_date,
        avg_trade_price,
        preco_medio_anterior,
        min_trade_price,
        max_trade_price,
        daily_trade_count,
        daily_usd_volume,
        variacao_absoluta_preco,
        variacao_percentual,
        modulo_variacao_percentual,
        quantidade_dias_historico,
        media_variacao_7d,
        desvio_variacao_7d,

        modulo_variacao_percentual - media_variacao_7d
            AS diferenca_variacao,

        SAFE_DIVIDE(
            modulo_variacao_percentual,
            media_variacao_7d
        ) AS razao_variacao,

        SAFE_DIVIDE(
            modulo_variacao_percentual - media_variacao_7d,
            desvio_variacao_7d
        ) AS z_score

    FROM estatisticas_moveis
),

classificacao AS (

    SELECT
        *,

        CASE
            WHEN quantidade_dias_historico < 7
                THEN 'Histórico insuficiente'

            WHEN desvio_variacao_7d IS NULL
                OR desvio_variacao_7d = 0
                THEN 'Sem variação histórica'

            WHEN z_score >= 4
                THEN 'Crítico'

            WHEN z_score >= 3
                THEN 'Alto'

            WHEN z_score >= 2
                THEN 'Moderado'

            ELSE 'Normal'
        END AS nivel_alerta

    FROM metricas
),

resultado_final AS (

    SELECT
        market_id,
        event_title,
        trade_date,

        ROUND(
            preco_medio_anterior,
            4
        ) AS preco_medio_anterior,

        ROUND(
            avg_trade_price,
            4
        ) AS preco_medio_atual,

        ROUND(
            variacao_absoluta_preco,
            4
        ) AS variacao_absoluta_preco,

        ROUND(
            variacao_percentual * 100,
            2
        ) AS variacao_percentual,

        ROUND(
            modulo_variacao_percentual * 100,
            2
        ) AS modulo_variacao_percentual,

        ROUND(
            media_variacao_7d * 100,
            2
        ) AS media_variacao_7d,

        ROUND(
            desvio_variacao_7d * 100,
            2
        ) AS desvio_variacao_7d,

        ROUND(
            diferenca_variacao * 100,
            2
        ) AS diferenca_variacao,

        ROUND(
            razao_variacao,
            2
        ) AS razao_variacao,

        ROUND(
            z_score,
            2
        ) AS z_score,

        CASE
            WHEN variacao_percentual > 0
                THEN 'Alta'

            WHEN variacao_percentual < 0
                THEN 'Queda'

            ELSE 'Estável'
        END AS direcao_variacao,

        nivel_alerta

    FROM classificacao
)

SELECT
    *

FROM resultado_final

WHERE nivel_alerta IN (
    'Moderado',
    'Alto',
    'Crítico'
)

ORDER BY
    z_score DESC,
    market_id,
    trade_date;
```
Que gerou `alerta_preco.csv`.

In [ ]:
alerta_preco = pd.read_csv("alerta_preco.csv")

alerta_preco["trade_date"] = pd.to_datetime(
    alerta_preco["trade_date"]
)

alerta_preco.head()

In [ ]:
print("Total de alertas:", len(alerta_preco))

print(
    "Mercados com alertas:",
    alerta_preco["market_id"].nunique()
)

print("\nAlertas por nível:")

display(
    alerta_preco["nivel_alerta"]
    .value_counts()
    .rename_axis("nivel_alerta")
    .reset_index(name="quantidade")
)

print("\nAlertas por direção:")

display(
    alerta_preco["direcao_variacao"]
    .value_counts()
    .rename_axis("direcao")
    .reset_index(name="quantidade")
)

In [ ]:
alertas_preco_por_mercado = (
    alerta_preco
    .groupby(["market_id", "event_title"])
    .size()
    .reset_index(name="quantidade_alertas")
    .sort_values(
        "quantidade_alertas",
        ascending=False
    )
)

display(alertas_preco_por_mercado.head(10))

In [ ]:
maiores_alertas_preco = (
    alerta_preco[
        [
            "market_id",
            "event_title",
            "trade_date",
            "preco_medio_anterior",
            "preco_medio_atual",
            "variacao_percentual",
            "direcao_variacao",
            "z_score",
            "nivel_alerta"
        ]
    ]
    .sort_values(
        "z_score",
        ascending=False
    )
    .head(10)
)

display(maiores_alertas_preco)

In [ ]:
alerta_preco["mes"] = (
    alerta_preco["trade_date"]
    .dt.to_period("M")
    .astype(str)
)

alertas_preco_por_mes = (
    alerta_preco
    .groupby("mes")
    .size()
    .reset_index(name="quantidade_alertas")
)

display(alertas_preco_por_mes)

#### Análise dos resultados

O segundo alerta identificou **892 variações anômalas de preço**, distribuídas entre **55 mercados**.

Entre as ocorrências identificadas, **644** representaram altas no preço médio diário e **248** representaram quedas.

O maior Score Z observado foi de **619.62**, no mercado **574072**, em **2025-12-27**. Nessa data, o preço médio passou de **0.0012** para **0.2914**, correspondendo a uma variação de **24185.27%**.

A utilização da variação percentual permite comparar mercados que apresentam níveis de preço diferentes. Além disso, o uso do módulo da variação faz com que tanto aumentos quanto reduções bruscas possam gerar alertas.

Como as estatísticas são calculadas separadamente por `market_id`, o limite de detecção se adapta ao comportamento recente de cada mercado.

As altas predominam sobre as quedas. Isso indica que, durante o período analisado, as maiores oscilações do preço médio ocorreram mais frequentemente em movimentos de alta do que de baixa.

Esse é um resultado interessante porque mostra uma assimetria no comportamento dos mercados analisados. Há muitos alertas classificados como críticos, indicando que diversas mudanças de preço foram muito superiores ao comportamento recente de cada mercado.



### Análise comparativa

In [ ]:
# Contagem de alertas do Alerta 1
volume = (
    alerta_volume
    .groupby("market_id")
    .size()
    .reset_index(name="alertas_volume")
)

# Contagem de alertas do Alerta 2
preco = (
    alerta_preco
    .groupby("market_id")
    .size()
    .reset_index(name="alertas_preco")
)

# Junta os resultados
comparacao = (
    volume
    .merge(preco, on="market_id", how="outer")
    .fillna(0)
)

display(comparacao.head())

plt.figure(figsize=(8,6))

plt.scatter(
    comparacao["alertas_volume"],
    comparacao["alertas_preco"]
)

plt.xlabel("Alertas de volume")
plt.ylabel("Alertas de preço")
plt.title("Comparação entre alertas de volume e preço por mercado")

plt.grid(True)

plt.show()

In [ ]:
comparacao = comparacao.sort_values(
    "alertas_volume",
    ascending=False
)

top20 = comparacao.head(20)

plt.figure(figsize=(14,6))

x = range(len(top20))

plt.bar(
    [i-0.2 for i in x],
    top20["alertas_volume"],
    width=0.4,
    label="Volume"
)

plt.bar(
    [i+0.2 for i in x],
    top20["alertas_preco"],
    width=0.4,
    label="Preço"
)

plt.xticks(
    x,
    top20["market_id"].astype(str),
    rotation=90
)

plt.xlabel("Market ID")
plt.ylabel("Quantidade de alertas")
plt.title("Comparação dos alertas por mercado")

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
correlacao = comparacao["alertas_volume"].corr(
    comparacao["alertas_preco"]
)

print(f"Correlação: {correlacao:.3f}")

A comparação entre os dois alertas mostrou que os mercados que apresentaram maior quantidade de alertas de volume também tendem a apresentar maior quantidade de alertas de preço.

O gráfico de barras evidencia que diversos `market_id` aparecem entre os mercados com maior número de ocorrências nos dois métodos. O gráfico de dispersão reforça essa observação, mostrando uma tendência crescente entre a quantidade de alertas de volume e de preço.

Além da análise visual, o coeficiente de correlação calculado entre as quantidades de alertas por mercado apresentou valor de **0.967**, indicando uma **forte correlação positiva** entre as duas variáveis.

Esse resultado sugere que mercados com atividade de negociação mais intensa também costumam apresentar maiores oscilações de preço. Embora a análise não permita concluir uma relação de causa e efeito, ela evidencia que os dois tipos de comportamento frequentemente ocorrem em conjunto nos mercados analisados.

## 5. Conclusão

Foram realizadas consultas exploratórias e analíticas sobre mercados da plataforma Polymarket, utilizando funções analíticas do SQL para identificar comportamentos atípicos.

O primeiro alerta detectou aumentos anormais no volume negociado, enquanto o segundo identificou variações incomuns no preço médio diário. Ambos utilizaram médias móveis, desvio padrão e Score Z calculados individualmente para cada `market_id`.

A análise dos resultados mostrou que os mercados relacionados ao evento "What price will Bitcoin hit in 2025?" concentraram mais alertas do que os mercados equivalentes de 2026. Também foi observada uma forte associação entre os dois alertas: mercados que apresentaram maior quantidade de aumentos anormais de volume também tenderam a apresentar maior quantidade de oscilações anômalas de preço. Essa conclusão foi sustentada pelos gráficos comparativos e pelo coeficiente de correlação obtido.

Os resultados indicam que a metodologia desenvolvida é capaz de identificar automaticamente comportamentos incomuns em diferentes mercados, sem a necessidade de definir limites fixos para volume ou preço. Dessa forma, as consultas construídas podem ser reutilizadas para monitorar outros mercados da plataforma, desde que possuam histórico suficiente para o cálculo das estatísticas móveis.